# ML-03 · Frame the question as an ML task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tessa-Saumu/FlyRank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

> **Freestyle submission.** This notebook belongs to my final, freestyle capstone: *which ranking pages are losing clicks to AI-answer search before it shows, and which should an editor protect first?* My earlier Lane 2 (refresh) experiments are kept for the record in `work/archive/` and are not part of this submission.

**What this card asks:** task type, target or proxy, success metric, the unit of analysis as a real dataframe, and why
ML beats a fixed rule. **The principle it serves:** decision first. Name the metric before training, and make the
target an observed outcome, never a rule's output.

## 1. Task type, target and metric (named before any training)

| Item | Choice | Why |
|---|---|---|
| Task type | supervised ranking: binary label, pages ranked by predicted probability | the decision is "which 50 first", not "yes or no for every page" |
| Target | `erode = 1` if clicks in months *o* and *o+1* < 0.75 x expected clicks | observed in a later window; nobody's rule defines it |
| Expected clicks | outcome impressions x current CTR (month *o-1*, shrunk) x position adjustment | compares the page with itself at its own rank |
| Primary metric | precision@50 per decision month, next to the base rate | 50 is the monthly review budget |
| Secondary | PR-AUC (whole ranking), precision@20/100, within-client precision@10 | P@50 alone has wide intervals (about +/-0.2 at K = 50) |
| Comparison | four frozen baselines and random order | the model must beat simple rules, not just chance |

## 2. The timeline (what is known when)

```
decision on day 1 of month o
features : months o-2 and o-1 (plus o-3 for one trend feature)   -> closed before the decision
label    : months o and o+1                                         -> observed after the decision
training : only origins <= o-2, whose labels have matured by month o-1
```

In [1]:
# Setup: works locally and in Colab. Installs only missing packages and never prints a token.
import importlib.util, json, os, sys
if any(importlib.util.find_spec(p) is None for p in ["duckdb", "huggingface_hub", "sklearn", "scipy", "matplotlib"]):
    get_ipython().run_line_magic("pip", "install -q duckdb huggingface_hub pandas scikit-learn scipy matplotlib")
here = os.path.abspath(os.getcwd())
ROOT = next((p for p in [here, os.path.dirname(here), os.path.dirname(os.path.dirname(here))]
             if os.path.isdir(os.path.join(p, "work", "scripts"))), None)
if ROOT is None:  # fresh Colab runtime: fetch the code (no data lives in git)
    get_ipython().system("git clone -q https://github.com/Tessa-Saumu/FlyRank-ML-Internship.git")
    ROOT = os.path.abspath("FlyRank-ML-Internship")
sys.path.insert(0, os.path.join(ROOT, "work", "scripts"))
OUT, FIG = os.path.join(ROOT, "work", "outputs"), os.path.join(ROOT, "work", "figures")
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings("ignore")
from IPython.display import Image, display, Markdown
import wh
DATA = wh.data_available(try_download=True)   # needs HF_TOKEN as an env var or Colab secret
print("pinned warehouse snapshot available:", DATA, "| revision", wh.REV[:8])
receipt = lambda name: json.load(open(os.path.join(OUT, name)))
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30)

pinned warehouse snapshot available: True | revision 50cbf7c3


## 3. The unit of analysis as a real dataframe

**Why show it:** a reviewer should see exactly what one row is. IDs are pseudonymous and used only for grouping, so they
are not shown. The frame is built by `work/scripts/erosion_frame.py` from the cached page-month table.

In [2]:
summ = receipt("erosion_frame_summary.json")
display(pd.DataFrame(summ["origins"]))
if DATA:
    import erosion_frame
    if not os.path.exists(wh.CACHE + "/erosion_frame.parquet"):
        erosion_frame.build_frame()
    f = pd.read_parquet(wh.CACHE + "/erosion_frame.parquet")
    cols = ["origin", "imp_f", "clk_f", "pos_f", "pos_m1", "ctr_now", "imp_out", "clk_out", "pos_out", "exp_out", "erode"]
    display(f.loc[f.origin == "2026-03", cols].sample(6, random_state=1).round(4))
    print("rows per origin:", f.groupby("origin").size().to_dict())

,origin,eligible,measurable,clients,base_rate,null_rate,lost_visibility,sealed,live_no_outcome
0,2025-12,5905,5378.0,19,0.3241,0.0515,0.0892,False,False
1,2026-01,6642,5785.0,19,0.2430,0.0455,0.1290,False,False
2,2026-02,7630,7083.0,21,0.2273,0.0517,0.0717,False,False
3,2026-03,8984,8613.0,29,0.3382,0.0513,0.0413,False,False
4,2026-04,12319,11186.0,30,0.3079,0.0580,0.0920,False,False
5,2026-05,13082,NaN,33,NaN,NaN,NaN,True,False
6,2026-07,14100,NaN,37,NaN,NaN,NaN,False,True


,origin,imp_f,clk_f,pos_f,pos_m1,ctr_now,imp_out,clk_out,pos_out,exp_out,erode
23188,2026-03,8126.0,20.0,5.4100,5.4100,0.0027,40196.0,117.0,4.2338,113.3047,0.0
24572,2026-03,15233.0,220.0,1.7504,1.7616,0.0134,17219.0,256.0,1.7495,231.2142,0.0
25273,2026-03,4403.0,34.0,7.5171,8.5452,0.0047,5281.0,15.0,9.2799,24.8892,1.0
22305,2026-03,37592.0,54.0,1.5625,1.6811,0.0017,102903.0,127.0,2.6339,177.2754,1.0
20508,2026-03,15972.0,167.0,8.3298,8.7619,0.0105,12728.0,99.0,18.8532,107.9379,0.0
26994,2026-03,52195.0,95.0,3.8329,3.8244,0.0018,242750.0,344.0,4.5362,409.1052,0.0


rows per origin: {'2025-12': 5905, '2026-01': 6642, '2026-02': 7630, '2026-03': 8984, '2026-04': 12319, '2026-05': 13082, '2026-07': 14100}


**Reading the table.** Each row is one top-10 page at one decision month, with at least 200 impressions and 20 clicks
in the two feature months. `exp_out` is what the page should earn in the next two months if its current CTR held at its
new position. `erode` compares what it actually earned against that expectation. The sealed origin (2026-05) and the
live origin (2026-07) have no base rate in the receipt: the first stays hidden until the end, the second has no outcome yet.

## 4. Three guards that make the label honest

| Guard | What it does | Why it is needed |
|---|---|---|
| Shrinkage (empirical Bayes, m = 1,000 impressions) | pulls month *o-1* CTR toward the client's position-adjusted CTR | at about 0.4% CTR, a page with 20 clicks has a noisy rate; shrinkage stops noise from posing as erosion |
| Fixed position curve (Oct-Nov 2025 only) | adjusts the expectation if rank moves | a page that drops rank loses clicks for a known reason; that is not AI-answer erosion |
| Poisson noise floor | simulates the label on pure sampling noise | shows how much of the base rate chance alone would produce |

The label anchors on the **latest** month, shrunk toward the **client**, not toward the page's own history. Two earlier
versions leaked momentum into the label. ML-05 (`w03_feature_leakage_check`) shows how I caught them.

In [3]:
o = pd.DataFrame(summ["origins"]).dropna(subset=["base_rate"])
o["signal_over_noise"] = (o.base_rate / o.null_rate).round(1)
display(o[["origin", "measurable", "clients", "base_rate", "null_rate", "signal_over_noise"]])

,origin,measurable,clients,base_rate,null_rate,signal_over_noise
0,2025-12,5378.0,19,0.3241,0.0515,6.3
1,2026-01,5785.0,19,0.2430,0.0455,5.3
2,2026-02,7083.0,21,0.2273,0.0517,4.4
3,2026-03,8613.0,29,0.3382,0.0513,6.6
4,2026-04,11186.0,30,0.3079,0.0580,5.3


**What it means.** Pure noise would label about 5% of pages as eroding. The observed rate is 23% to 34%, four to seven
times higher, so most positives are real changes in click share at a stable rank, not counting noise.

## 5. Why ML instead of a fixed rule

- **Erosion is a combination, not one signal.** Rank, a recent CTR bump, CTR above what the position normally earns,
  demand swings and client-wide shifts all matter together (ML-06, ML-08).
- **The intuitive rules point the wrong way.** In the training months, top-3 pages eroded *less* than positions 7 to 10,
  and pages whose CTR had just risen eroded *more* than pages already slipping (ML-07).
- **The best simple rule changes every month** (ML-08), so no single if-statement holds.

The output feeds a real content action. ML-10 maps each reason code to an editor action, with human review before any
change.